In [0]:
# Intenção: parâmetros da ingestão (alterar ANO/MES a cada execução)
import base64

PARENT_PROJECT = "lakehouse-cnes"
PROJECT_ID = "basedosdados"
DATASET = "br_ms_cnes"
TABLE = "estabelecimento"
UF = "RJ"
ANO = 2025
MES = 12
VOLUME_PATH = "/Volumes/lakehouse_cnes/bronze/raw_files/estabelecimento"
DESTINO = f"{VOLUME_PATH}/{ANO}_{MES:02d}"
FILTRO_BQ = f"ano = {ANO} AND mes = {MES} AND sigla_uf = '{UF}'"

In [0]:
# Intenção: ler a competência no BigQuery com filtro na origem e um único stream
# (múltiplos streams perdiam linhas quando a leitura incluía muitas colunas)
credentials_json = dbutils.secrets.get(
    scope="bigquery",
    key="bigquery-service-account"
)

credentials_base64 = base64.b64encode(
    credentials_json.encode("utf-8")
).decode("ascii")

df = (
    spark.read
        .format("bigquery")
        .option("credentials", credentials_base64)
        .option("parentProject", PARENT_PROJECT)
        .option("project", PROJECT_ID)
        .option("dataset", DATASET)
        .option("table", TABLE)
        .option("filter", FILTRO_BQ)
        .option("maxParallelism", "1")
        .load()
)

# Conclusão esperada: total de registros da competência segundo a fonte
ESPERADO = df.count()
print("Registros na fonte:", ESPERADO)

if ESPERADO == 0:
    raise ValueError(f"Competência {ANO}_{MES:02d} sem registros na fonte (indisponível ou restrita ao BD Pro). Nada será gravado.")

In [0]:
# Intenção: gravar a competência em JSON e validar a completude contra a fonte
try:
    dbutils.fs.ls(DESTINO)
    print(f"Competência {ANO}_{MES:02d} já existe. Nada foi gravado.")
except Exception:
    (
        df
        .coalesce(1)
        .write
        .json(DESTINO)
    )

    GRAVADO = spark.read.json(DESTINO).count()

    if GRAVADO != ESPERADO:
        dbutils.fs.rm(DESTINO, recurse=True)
        raise ValueError(f"Falha de completude: fonte={ESPERADO}, arquivo={GRAVADO}. Pasta removida.")

    print(f"Competência {ANO}_{MES:02d} gravada e validada: {GRAVADO} registros")

# Conclusão esperada: mensagem de validação com o mesmo número da fonte